# G3 - Heat transfer: $h_1$, $h_2$, $U_2$

**057922 Nuclear Design and Technology** · Project Lab

Author: Giovanni Zullo

---

## In this notebook

The three coefficients your temperature field needs, first on neutral data: the inner film
by forced convection, the outer by natural convection, and the overall outer coefficient
that combines the insulation and the outer film. Then the same three for your own vessel.

## Why the range check matters

Because the answer is not a number, it is a number **with a validity statement**. Every
correlation you are about to use was fitted to a range of Reynolds, Prandtl and geometry.
Outside that range it still returns a number, with no warning, and that
number is not evidence of anything.

The professional habit to build here: **compute the coefficient, then check the range,
then state the check**. When you are outside, you say so and you say what you did about it.

---
## Step 0: Environment

In [1]:
# --- where is the course material? (Colab only; a no-op on your own machine) -------
# Nothing to configure. Three places are tried, in the order that costs least:
#   1. already on disk next to this notebook  -> nothing to do
#   2. fetchable from GitHub                  -> the next cell downloads it
#   3. neither                                -> mount Google Drive and look there
# Drive is the LAST resort on purpose: when the course repository is public, step 2
# succeeds and nobody is ever asked to authorise Drive. When it is not -- a private
# repo, or no network -- step 3 is exactly what is needed, so it happens by itself.
import os, glob, urllib.request

URL = ('https://raw.githubusercontent.com/giozu/ndt-labs/'
       'main/launcher/ndt_setup.py')

def _on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _find_labs(start=None):
    # The course folder above us, or None. Same walk-up the bootstrap uses.
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, 'launcher', 'ndt_setup.py')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

def _github_has_it():
    try:
        req = urllib.request.Request(URL, method='HEAD')
        with urllib.request.urlopen(req, timeout=8) as r:
            return r.status == 200
    except Exception:
        return False

def _search_drive():
    # The obvious place first -- the course folder uploaded to the top of My Drive --
    # then one or two folders down, in case it was tidied away. NOT a deep recursive
    # hunt: if it is not within reach of these, the problem is not the depth.
    D = '/content/drive/MyDrive'
    for pat in (f'{D}/ndt-labs/launcher/ndt_setup.py',
                f'{D}/*/launcher/ndt_setup.py',
                f'{D}/*/*/launcher/ndt_setup.py'):
        hits = glob.glob(pat)
        if hits:
            return hits[0]
    return None


if not _on_colab():
    print('[where] local machine - the files are already here')
elif _find_labs():
    print('[where] already here:', _find_labs())
elif _github_has_it():
    print('[where] will be downloaded from GitHub by the next cell')
else:
    print('[where] not on disk and not reachable on GitHub - looking on Google Drive')
    from google.colab import drive
    drive.mount('/content/drive')

    # A mount can succeed and still show NOTHING: a stale mount left by an earlier
    # attempt, or the wrong Google account. That looked exactly like "the folder is
    # somewhere unexpected" once, and sent the search deeper and deeper for nothing.
    # Distinguish the two, because the remedies have nothing in common.
    top = glob.glob('/content/drive/MyDrive/*')
    if not top:
        raise SystemExit(
            'Google Drive is mounted but MyDrive looks EMPTY.\n'
            'That is a mount problem, not a missing folder. Either an earlier mount is\n'
            'stale, or you authorised a different Google account. Fix it with:\n'
            "    drive.mount('/content/drive', force_remount=True)\n"
            'and pick the account that holds the course folder. If it stays empty,\n'
            'Runtime -> Disconnect and delete runtime, then start again.')

    hit = _search_drive()
    if not hit:
        raise SystemExit(
            'Your Drive is readable but no launcher/ndt_setup.py was found in it.\n'
            'Top level of MyDrive:\n  ' + '\n  '.join(sorted(top)[:20]) + '\n'
            'Upload the course folder there, or os.chdir() into it by hand.')

    # The repository ROOT is enough: everything downstream walks up from the working
    # directory, so there is no per-notebook path to keep in sync here.
    os.chdir(os.path.dirname(os.path.dirname(hit)))
    print('[where] found on Drive:', os.getcwd())

print('cwd      :', os.getcwd())

[where] local machine - the files are already here
cwd      : ~/ndt-labs/sessions/G03_heat_transfer


In [2]:
import os, sys, subprocess

# Get the shared bootstrap, ndt_setup.py. Two ways, and the notebook picks one:
#  - LOCAL: you cloned ndt-labs, so launcher/ndt_setup.py is already on disk a few
#    directories up. Use that copy -- no network, and any pin you edit takes effect.
#  - COLAB: fetch it. Check the file is NON-EMPTY: wget -O creates
#    a zero-byte file when the download fails, that file "exists", the download is then
#    skipped forever, and `import ndt_setup` silently imports nothing.
URL = ('https://raw.githubusercontent.com/giozu/ndt-labs/'
       'main/launcher/ndt_setup.py')

def _find_launcher(d=None):
    d = os.path.abspath(d or os.getcwd())
    while True:
        p = os.path.join(d, 'launcher', 'ndt_setup.py')
        if os.path.isfile(p) and os.path.getsize(p) > 0:
            return os.path.dirname(p)
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

launcher = _find_launcher()
if launcher is None:
    if not os.path.exists('ndt_setup.py') or os.path.getsize('ndt_setup.py') == 0:
        subprocess.run(['wget', '-q', URL, '-O', 'ndt_setup.py'])
    if not os.path.exists('ndt_setup.py') or os.path.getsize('ndt_setup.py') == 0:
        raise SystemExit('bootstrap download failed (0 bytes). Check network access '
                         'to the course repository, then re-run this cell.')
    launcher = os.getcwd()
sys.path.insert(0, launcher)

import ndt_setup
ndt_setup.setup()

[ndt] local environment detected -- nothing will be installed
  python   ~/miniconda3/envs/z3st/bin/python3.12


  z3st at  ~/z3st
  labs at  ~/ndt-labs

--- environment check ---


  dolfinx  0.11.0
  numpy    2.4.6


  gmsh     4.15.2
  z3st     0.3.2   [course-2627.1]
  MPI      Open MPI v5.0.10, package: Open MPI conda@9ee00b64ad27 

ENVIRONMENT OK


True

---
## 1. The inner film: forced convection

The coolant is pumped, so this is forced convection, and the workhorse correlation is
**Dittus-Boelter**:

$$\mathrm{Nu} = 0.023\,\mathrm{Re}^{0.8}\,\mathrm{Pr}^{n},
\qquad n = 0.4 \ \text{(wall heats the fluid)},\quad n = 0.3 \ \text{(wall cools it)}$$

Two things students get wrong here and both cost real accuracy:

- **The exponent.** Here the wall is hotter than the coolant (gamma heating in the wall,
  remember), so the wall *heats* the fluid and $n = 0.4$.
- **The characteristic length.** For an annular downcomer the hydraulic diameter is
  $D_h = 4A/P = 2\times(\text{gap})$, *not* the vessel diameter. Using the vessel
  diameter inflates Re by a factor of about six and quietly breaks the entrance-length
  criterion.

By hand, with $\dot m$ the mass flow rate in the downcomer:

$$u = \frac{\dot m}{\rho A}, \qquad A = \pi (R_{wet}^2 - R_{barrel}^2), \qquad
\mathrm{Re} = \frac{\rho u D_h}{\mu}, \qquad \mathrm{Pr} = \frac{\mu c_p}{k}, \qquad
h_1 = \frac{\mathrm{Nu}\, k}{D_h}$$

and the three ranges to check: $\mathrm{Re} > 10^4$, $0.6 < \mathrm{Pr} < 160$, $L/D_h > 10$.

In [3]:
import numpy as np
import matplotlib.pyplot as plt

# --- pressurised water at ~560 K, 15 MPa (rounded, neutral) --------------------
rho, cp, k_w, mu_w = 750.0, 5300.0, 0.58, 9.5e-5
m_dot, L = 319.27, 2.0               # mass flow rate (kg/s), vessel height (m)
R_barrel, R_wet = 0.246, 0.296       # barrel outer radius, wetted surface (m)

gap = R_wet - R_barrel               # width of the annular downcomer
A = np.pi*(R_wet**2 - R_barrel**2)   # flow area of the annulus
u = m_dot/(rho*A)                    # velocity, from the mass flow rate
Dh = 2*gap                           # 4A/P of an annulus

print(f"gap = {gap*1e3:.1f} mm    A = {A:.4f} m2    u = {u:.3f} m/s    Dh = {Dh:.3f} m")

Pr = mu_w*cp/k_w
Re = rho*u*Dh/mu_w
Nu = 0.023*Re**0.8*Pr**0.4
h1 = Nu*k_w/Dh

print(f"Pr = {Pr:.3f}    Re = {Re:.3e}    Nu = {Nu:.1f}")
print(f"h1 = {h1:.0f} W/m2K\n")

print("Validity of Dittus-Boelter: CHECK, do not assume.")
checks = [("Re  > 1e4 (fully turbulent)", Re, 1e4, np.inf),
          ("0.6 < Pr < 160", Pr, 0.6, 160.0),
          ("L/Dh > 10 (entrance effects)", L/Dh, 10.0, np.inf)]
for lab, val, lo, hi in checks:
    ok = lo <= val <= hi
    print(f"  {lab:32} {val:10.4g}   {'OK' if ok else '*** OUT OF RANGE ***'}")
print("\nHad we used the vessel diameter instead of the gap:")
Dh_wrong = 2*R_wet                   # the vessel diameter
print(f"  Dh = {Dh_wrong:.3f} m -> Re = {rho*u*Dh_wrong/mu_w:.2e}, "
      f"L/Dh = {L/Dh_wrong:.1f}  *** entrance criterion VIOLATED ***")

gap = 50.0 mm    A = 0.0851 m2    u = 5.000 m/s    Dh = 0.100 m
Pr = 0.868    Re = 3.947e+06    Nu = 4113.5
h1 = 23858 W/m2K

Validity of Dittus-Boelter: CHECK, do not assume.
  Re  > 1e4 (fully turbulent)       3.947e+06   OK
  0.6 < Pr < 160                       0.8681   OK
  L/Dh > 10 (entrance effects)             20   OK

Had we used the vessel diameter instead of the gap:
  Dh = 0.592 m -> Re = 2.34e+07, L/Dh = 3.4  *** entrance criterion VIOLATED ***


---
## 2. The outer film: natural convection

Outside there is no pump. Flow is driven by the density difference the hot wall creates,
so the governing group is Rayleigh, and for a tall vertical surface **McAdams** gives

$$\mathrm{Nu} = 0.13\,\mathrm{Ra}^{1/3} \qquad (\mathrm{Ra} > 10^9,\ \text{turbulent})$$

$$\mathrm{Gr} = \frac{g \beta \Delta T L^3 \rho^2}{\mu^2}, \qquad
\mathrm{Ra} = \mathrm{Gr}\,\mathrm{Pr}, \qquad h_2 = \frac{\mathrm{Nu}\, k}{L}$$

with the properties of the containment water.

This one has a feature worth understanding rather than working around: $\mathrm{Ra}$
contains $\Delta T = T_{\text{wall}} - T_\infty$, where the "wall" is the outer face of the
insulation, in contact with the containment water, and its temperature is what you are
trying to find. **In general the problem is implicit.** If the data fix the $\Delta T$ for Gr, use that
value and state it; the sensitivity below tells you what that choice is worth.

And a pleasant consequence of the $1/3$ exponent: $\mathrm{Ra}\propto L^3$ while
$\mathrm{Nu}\propto \mathrm{Ra}^{1/3}$, so **$h_2$ does not depend on the height at all**.
That is what makes it safe to apply a vertical-plate correlation to a vessel whose height
you have not fixed yet.

In [4]:
# --- containment water, still, film properties around 322 K (rounded, neutral) ----
T_inf = 320.0
g_acc, beta = 9.81, 4.3e-4
rho_c, cp_c, k_c, mu_c = 989.0, 4180.0, 0.64, 5.8e-4
Pr_c = mu_c*cp_c/k_c

def h2_of(T_wall, L=2.0):
    dT = max(T_wall - T_inf, 1e-6)
    Ra = g_acc*beta*dT*L**3*rho_c**2/mu_c**2*Pr_c
    return 0.13*Ra**(1/3)*k_c/L, Ra

T_guess = 325.0      # outer face of the insulation: a few K above the water
h2, Ra = h2_of(T_guess)
print(f"Pr(water) = {Pr_c:.3f}")
print(f"assuming T_wall = {T_guess:.0f} K:  Ra = {Ra:.3e}   h2 = {h2:.0f} W/m2K")
print(f"  turbulent branch needs 1e9 < Ra < 1e13   ->  {'OK' if 1e9 < Ra < 1e13 else 'OUT OF RANGE'}\n")

print("h2 is independent of the height -- Ra ~ L^3 and Nu ~ Ra^(1/3) cancel exactly:")
for LL in (1.0, 2.0, 5.0, 10.0):
    print(f"   L = {LL:5.1f} m  ->  h2 = {h2_of(T_guess, LL)[0]:.1f} W/m2K")

print("\nand it is sensitive to the wall temperature you assumed:")
for Tw in (321.0, 322.0, 325.0, 330.0, 340.0):
    print(f"   T_wall = {Tw:5.0f} K  ->  h2 = {h2_of(Tw)[0]:6.1f} W/m2K")
print(f"   (h2 ~ dT^(1/3): guessing 5 K above the water when the truth is 1 K"
      f" overestimates h2 by a factor {h2_of(325.0)[0]/h2_of(321.0)[0]:.1f}.)")

Pr(water) = 3.788
assuming T_wall = 325 K:  Ra = 1.858e+12   h2 = 511 W/m2K
  turbulent branch needs 1e9 < Ra < 1e13   ->  OK

h2 is independent of the height -- Ra ~ L^3 and Nu ~ Ra^(1/3) cancel exactly:
   L =   1.0 m  ->  h2 = 511.5 W/m2K
   L =   2.0 m  ->  h2 = 511.5 W/m2K
   L =   5.0 m  ->  h2 = 511.5 W/m2K
   L =  10.0 m  ->  h2 = 511.5 W/m2K

and it is sensitive to the wall temperature you assumed:
   T_wall =   321 K  ->  h2 =  299.1 W/m2K
   T_wall =   322 K  ->  h2 =  376.8 W/m2K
   T_wall =   325 K  ->  h2 =  511.5 W/m2K
   T_wall =   330 K  ->  h2 =  644.4 W/m2K
   T_wall =   340 K  ->  h2 =  811.9 W/m2K
   (h2 ~ dT^(1/3): guessing 5 K above the water when the truth is 1 K overestimates h2 by a factor 1.7.)


---
## 3. $U_2$: combining them, referred to a stated surface

The outer boundary condition your thermal model needs is a single overall coefficient
between the vessel outer surface and the containment water. It is a series of
resistances, insulation and outer film, and in cylindrical geometry each must be
**referred to the same radius**, because the area changes with $r$.

$$\frac{1}{U_2} = \underbrace{\frac{R_o\ln(R_2/R_o)}{k_{\text{ins}}}}_{\text{insulation}}
+ \underbrace{\frac{R_o}{R_2\,h_2}}_{\text{film}}$$

The steel wall is **not** in it. The wall carries the gamma source, so it is the domain
your thermal model solves, not a resistance in its boundary condition. $U_2$ is applied
at $r = R_o$, and conduction through the steel is the job of the model. Put the wall in $U_2$
as well and you count it twice.

**A $U$ without a stated reference surface is meaningless.** The film acts on the outer
face of the insulation, $r = R_2$, and the factor $R_o/R_2$ moves it to the vessel
surface. Write "$U_2$ referred to $r = R_o$" every time.

In [5]:
t_L, t_v = 0.004, 0.100               # liner and vessel thickness (m): state yours, even if tentative
Ro = R_wet + t_L + t_v                # vessel outer radius: where U2 is applied
t_ins, k_ins = 0.050, 0.060           # 50 mm of generic mineral wool
R2 = Ro + t_ins                       # outer face of the insulation

# both resistances per unit area of the VESSEL OUTER SURFACE, r = Ro
R_ins  = Ro*np.log(R2/Ro)/k_ins
R_film = Ro/(R2*h2)
R_tot  = R_ins + R_film
U2 = 1.0/R_tot

print(f"{'resistance':14}{'m2K/W':>10}{'share':>9}")
for lab, R in (('insulation', R_ins), ('outer film', R_film)):
    print(f"{lab:14}{R:10.4f}{100*R/R_tot:8.1f} %")
print(f"{'TOTAL':14}{R_tot:10.4f}")
print(f"\nU2 = {U2:.3f} W/m2K, referred to r = Ro = {Ro:.3f} m\n")

share_ins = R_ins/R_tot
dominant = 'insulation' if share_ins > 0.5 else 'outer film'
print(f"Sensitivity: the {dominant} carries {100*max(share_ins, 1 - share_ins):.1f} % of the resistance.")
U2_of = lambda f: 1/(R_ins + Ro/(R2*f*h2))
for f in (0.5, 0.75, 1.0, 1.5, 2.0):
    print(f"   h2 x {f:4.2f} ({f*h2:6.1f} W/m2K)  ->  U2 = {U2_of(f):.4f} W/m2K")
print("\nThat is the answer to 'my correlation is out of range': find out whether the")
print(f"quantity it feeds is even sensitive to it. Here a factor 4 on h2 (x0.5 to x2) moves U2")
print(f"by {100*(U2_of(2.0)/U2_of(0.5) - 1):.2f} %. Whether U2 matters to the wall temperature is checked in G4.")

resistance         m2K/W    share
insulation        0.7852    99.8 %
outer film        0.0017     0.2 %
TOTAL             0.7870

U2 = 1.271 W/m2K, referred to r = Ro = 0.400 m

Sensitivity: the insulation carries 99.8 % of the resistance.
   h2 x 0.50 ( 255.7 W/m2K)  ->  U2 = 1.2679 W/m2K
   h2 x 0.75 ( 383.6 W/m2K)  ->  U2 = 1.2698 W/m2K
   h2 x 1.00 ( 511.5 W/m2K)  ->  U2 = 1.2707 W/m2K
   h2 x 1.50 ( 767.2 W/m2K)  ->  U2 = 1.2717 W/m2K
   h2 x 2.00 (1022.9 W/m2K)  ->  U2 = 1.2721 W/m2K

That is the answer to 'my correlation is out of range': find out whether the
quantity it feeds is even sensitive to it. Here a factor 4 on h2 (x0.5 to x2) moves U2
by 0.33 %. Whether U2 matters to the wall temperature is checked in G4.


---
## 4. Your vessel

Write your data in the cell below and run it: the neutral cases above stay as they are, for comparison.

1. **$h_1$**: your coolant, your mass flow rate, your $D_h$ (the *gap*, not the diameter),
   your exponent (is the wall heating or cooling the fluid?). Check all three ranges.
2. **$h_2$**: your outer environment. If your data fix the $\Delta T$ for Gr, use it
   and state it; the sensitivity of §2 tells you what that choice is worth.
3. **$U_2$**: with your insulation and your (tentative) thicknesses, **referred to the vessel outer surface**, with
   the steel wall left out of it.
4. **Where you are out of range**, if anywhere, and what you did: tightened the geometry,
   used a different correlation, or carried the uncertainty forward and showed it does not
   matter.

### For your report

$h_1$, $h_2$, $U_2$ with their validity checks.

### Checklist

- Three coefficients, each with a range check you actually ran.
- One sentence naming the dominant resistance in your $U_2$, and what that implies about
  which of your inputs deserves care.

In [6]:
# ---- your vessel ---------------------------------------------------------------
MY_rho, MY_cp, MY_k, MY_mu = 0.0, 0.0, 0.0, 0.0     # your coolant properties
MY_u, MY_gap = 0.0, 0.0                             # velocity, downcomer gap
# MY_Dh = 2*MY_gap
# MY_Pr = MY_mu*MY_cp/MY_k ; MY_Re = MY_rho*MY_u*MY_Dh/MY_mu
# MY_h1 = 0.023*MY_Re**0.8*MY_Pr**0.4*MY_k/MY_Dh
print('Template cell: fill in your own numbers.')

Template cell: fill in your own numbers.
